# Dataset Preparation

This notebook brings the original datsets into the necessary format for this thesis and creates the lexical extension to the dataset.

In [1]:
import pandas as pd
import random
import os
from datasets import load_dataset
import spacy
from spacy.tokens import Doc
from nltk.corpus import wordnet as wn
from nltk.corpus import gutenberg, inaugural, brown
import json
import csv
import nltk
from modules.evaluation import load_glitter, load_gente, load_lexical, load_fairtranslate, small_enough_for_spacy, load_corpus_sources, clean_sentence

/home/marie-necker/anaconda3/envs/coralreef/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
nlp = spacy.load("en_core_web_md")

In [3]:
#os.makedirs("../data/glitter-splits", exist_ok=True)
#os.makedirs("../data/fairtranslate-splits", exist_ok=True)
#os.makedirs("../data/gente-splits", exist_ok=True)
#os.makedirs("../data/lexical-splits", exist_ok=True)

In [4]:
glitter_path = '../../data/original-datasets/glitter.parquet'
glitter_dev = '../../data/glitter-splits/glitter-dev.tsv'
glitter_test = '../../data/glitter-splits/glitter-test.tsv'
glitter_train = '../../data/glitter-splits/glitter-train.tsv'

fairtranslate_dataset = load_dataset("Fannyjrd/FairTranslate_fr")
fairtranslate_dev = '../../data/fairtranslate-splits/fairtranslate-dev.tsv'
fairtranslate_test = '../../data/fairtranslate-splits/fairtranslate-test.tsv'
fairtranslate_train = '../../data/fairtranslate-splits/fairtranslate-train.tsv'

gente_path = '../../data/original-datasets/GeNTE/GeNTE.tsv'
gente_dev = '../../data/gente-splits/gente-dev.tsv'
gente_test = '../../data/gente-splits/gente-test.tsv'
gente_train = '../../data/gente-splits/gente-train.tsv'
gente_full = '../../data/gente-splits/gente-full.tsv'

female_lexical = '../../data/lexical-splits/lexical-female.tsv'
male_lexical = '../../data/lexical-splits/lexical-male.tsv'
lexical_dev = '../../data/lexical-splits/lexical-dev.tsv'
lexical_test = '../../data/lexical-splits/lexical-test.tsv'
lexical_train = '../../data/lexical-splits/lexical-train.tsv'

In [5]:
def write_to_tsv(items, outfile_path):
     with open(outfile_path, 'w', encoding='utf8') as outfile:
        for item in items:
            outfile.write('\t'.join(item) + '\n')
        outfile.write('\n')

## Glitter

In [6]:
#code in this cell taken/adapted from https://github.com/pranav-ust/glitter

df = pd.read_parquet(glitter_path)

english_sources = df['preceding_context'] + ' ' + df['matching_sentence'] + ' ' + df['trailing_context']
german_references = {
    'neutral': df['neutral_PE'],
    'gender_star': df['gender-star_PE'],
    'ens_forms': df['ens_PE']
    }

ambiguity_types = df['ambiguity'].value_counts()

df.to_csv('dataset.tsv', sep='\t', index=False)

In [7]:
print(ambiguity_types)

ambiguity
ambiguous             501
unambiguous_male      211
unambiguous_female    183
unambiguous_all        85
Name: count, dtype: int64


In [8]:
random.seed(14)

probier = []
probier2 = []
probier3 = []
for seed, pre, match, trail, gold in zip(df['seed'], df['preceding_context'], df['matching_sentence'], df['trailing_context'], df['ambiguity']):
    probier.append([seed, pre, match, trail, gold])
#print(len(probier))
#print(probier[4])

random.seed(14)
dev_sample = random.sample(probier, int(len(probier) * 0.1))   # dev: 10%
#print(dev_sample[0])

#for item in dev_sample:
 #   print(item)
  #  print()

random.shuffle(probier)

for item in probier:
    if item not in dev_sample:
        probier2.append(item)
test_sample = random.sample(probier2, int(len(probier) * 0.4))  # test: 40%


for item in probier:
    if item not in dev_sample and item not in test_sample:
        probier3.append(item)
train_sample = probier3
random.shuffle(train_sample)

# train: 50%

print('Length Dev Set:',len(dev_sample))
#print()
print('Length Test Set:',len(test_sample))
#print()
print('Length Train Set:',len(train_sample))
print()
print('Length Complete Examples:', len(dev_sample) + len(test_sample) + len(train_sample))
write_to_tsv(dev_sample, glitter_dev)
write_to_tsv(test_sample, glitter_test)
write_to_tsv(train_sample, glitter_train)

Length Dev Set: 98
Length Test Set: 392
Length Train Set: 490

Length Complete Examples: 980


## FairTranslate

Since FairTranslate does not contain information on the English seed nouns (only the french translations), which are the relevant nouns for gender detection, they had to be extracted first.

In [9]:
#french seed nouns were translated into English (several different options per noun) to get seed nouns using Claude Sonnet 4.6
#french seeds: https://github.com/fanny-jourdan/FairTranslate/blob/main/data/occupations_lists.py
#all titles (or in one case: names) that were not in the list were added manually to be able to automatically mark seed nouns in the data set
#female and male signify stereotypical associations in this case, not lexical gender.


occupations_female = [
    ["dietitian", "nutritionist", "dietary aide", "dietician"],
    ["cleaner", "housekeeper", "custodian"],
    ["teacher", "schoolteacher", "instructor"],
    ["childminder", "nanny", "caregiver"],
    ["care aide", "care assistant", "nursing aide", "assistant"],
    ["nurse", "registered nurse", "nursing professional"],
    ["pharmacy assistant", "pharmacy technician", "pharmacy aide"],
    ["hairdresser", "stylist", "barber", "hairstylist"],
    ["beautician", "esthetician", "cosmetologist"],
    ["cashier", "checkout clerk", "teller"],
    ["counter clerk", "window clerk", "ticket agent"],
    ["accountant", "bookkeeper", "accounting clerk", "accounting employee"],
    ["social worker", "caseworker", "counselor"],
    ["pharmacist", "druggist", "dispensary professional"],
    ["salesperson", "vendor", "sales clerk"],
    ["flight attendant", "cabin crew", "steward"],
    ["childcare nurse", "pediatric nurse", "nursery nurse"],
    ["home aide", "personal assistant", "care assistant"],
    ["daycare worker", "nursery worker", "childcare worker"],
    ["housekeeper", "governess", "household manager"],
    ["secretary", "administrator", "clerk"],
    ["librarian", "archivist", "information specialist"],
]

occupations_male = [
    ["construction worker", "builder", "laborer"],
    ["lumberjack", "logger", "forester"],
    ["firefighter", "fire officer", "rescuer"],
    ["electrician", "wiring technician", "lineman"],
    ["welder", "metal worker", "solderer"],
    ["plumber", "pipefitter", "pipe worker"],
    ["mechanic", "technician", "repair person"],
    ["carpenter", "framer", "joiner"],
    ["woodworker", "joiner", "cabinetmaker"],
    ["electromechanical technician", "maintenance technician", "electromechanic"],
    ["street sweeper", "sanitation worker", "janitor"],
    ["garbage collector", "refuse collector", "sanitation worker"],
    ["butcher", "meat cutter", "meatpacker"],
    ["engineer", "developer", "designer", "computer scientist"],
    ["bus driver", "transit driver", "coach driver"],
    ["foreman", "supervisor", "overseer"],
    ["IT specialist", "computer technician", "systems analyst"],
    ["programmer", "developer", "coder"],
    ["police officer", "cop", "law enforcement officer"],
    ["surgeon", "operating physician", "specialist"],
    ["crane operator", "machine operator", "equipment operator"],
    ["upholsterer", "furniture maker", "upholstery worker"],
]

occupations_neutral = [
    ["chemistry technician", "lab technician", "chemical analyst"],
    ["management controller", "financial controller", "budget analyst"],
    ["press operator", "machine operator", "print operator"],
    ["buyer", "purchaser", "procurement officer"],
    ["quality technician", "quality inspector", "quality controller"],
    ["physiotherapist", "physical therapist", "rehabilitation therapist"],
    ["lawyer", "attorney", "counsel"],
    ["waiter", "server", "food service worker"],
    ["teacher", "educator", "instructor"],
    ["product manager", "brand manager", "product owner"],
    ["translator", "interpreter", "language specialist"],
    ["project officer", "coordinator", "program officer"],
    ["career counselor", "guidance counselor", "orientation advisor"],
    ["physician", "doctor", "medical professional"],
    ["optician", "eyewear specialist", "optical dispenser"],
    ["special needs educator", "support worker", "inclusion specialist"],
    ["journalist", "reporter", "correspondent"],
    ["accountant", "bookkeeper", "financial clerk"],
    ["Morgan"],
]

In [10]:
#extract seed nouns from sentences


occupation_titles_translated = occupations_female + occupations_male + occupations_neutral
seed_nouns = list()
sentences_jourdan = list()

for sent in fairtranslate_dataset['train']['english']:
    sentences_jourdan.append(sent)
    spacy_sent = nlp(sent)
    current_occupation = None
    found = False
    for token in spacy_sent:
        for occupation in occupation_titles_translated:
            matched = False
            for version in occupation:
                if token.text == version:
                    #print(token.text)
                    current_occupation = token.text
                    seed_nouns.append(current_occupation)
                    matched = True
                    break
                elif len(version.split(' ')) > 1 and token.text == version.split(' ')[-1]:
                    #print(token.text, '2nd route')
                    current_occupation = token.text
                    seed_nouns.append(current_occupation)
                    matched = True
                    break
            if matched:
                found = True
                break
        if found:
            break
    if current_occupation == None:
        #print('that must be a special one')
        #print(sent)
        seed_nouns.append(sent)


unique_seeds = list(set(seed_nouns))
#print(unique_occupations)

In [11]:
print(unique_seeds)

['salesperson', 'butcher', 'officer', 'mechanic', 'joiner', 'operator', 'teacher', 'journalist', 'physiotherapist', 'upholsterer', 'carpenter', 'server', 'hairstylist', 'surgeon', 'dietician', 'pharmacist', 'driver', 'engineer', 'attendant', 'therapist', 'employee', 'Morgan', 'hairdresser', 'cleaner', 'librarian', 'lawyer', 'collector', 'housekeeper', 'plumber', 'buyer', 'administrator', 'nurse', 'counselor', 'beautician', 'educator', 'specialist', 'translator', 'manager', 'programmer', 'supervisor', 'welder', 'firefighter', 'optician', 'accountant', 'controller', 'secretary', 'logger', 'electromechanic', 'doctor', 'childminder', 'sweeper', 'schoolteacher', 'teller', 'cashier', 'dietitian', 'scientist', 'assistant', 'worker', 'electrician', 'caregiver', 'technician']


In [12]:
#print(dataset['train'][0])

dataset_i_need = list()

for example in fairtranslate_dataset['train']:
    dict_i_need = dict()
    ambiguity = example['ambiguity']
    dict_i_need['ambiguity'] = ambiguity
    gender = example['gender']
    dict_i_need['gender'] = gender
    stereotype = example['stereotype']
    dict_i_need['stereotype'] = stereotype
    sentence = example['english']
    dict_i_need['sentence'] = sentence
    for token in nlp(sentence):
        if token.text in unique_seeds:
            seed = token.text
            dict_i_need['seed'] = seed
    
    dataset_i_need.append(dict_i_need)

print(dataset_i_need[75])

{'ambiguity': 'ambiguous', 'gender': 'male', 'stereotype': 'female-stereotyped', 'sentence': 'A nurse prepares medical files before a routine check-up.', 'seed': 'nurse'}


The gender 'male' in the previous example refers to the French translation. The ambiguity label (ambiguous or unambiguous) indicates whether the gender can be deduced from the English source text. The next step will therefore combine the ambiguity and gender labels to get the gender label relevant to linguistic gender detection from English text. All examples labeled as 'ambiguous' will retain that label while all examples labeled as 'unambiguous' receive a label that is the combination of ambiguity and gender (unambiguous_female, unambiguous_male).

In [13]:
new_dict_i_need = dict()
dataset_i_really_need = []
unique_sents = set()

for entry in dataset_i_need:
    new_dict_i_need = dict()
    sent = entry['sentence']
    ambiguity = entry['ambiguity']
    gender = entry['gender']

    if sent in unique_sents:     # some sentences appear multiple times for different translations (not relevant to this task)
        continue 

    if ambiguity == 'ambiguous':
        label = 'ambiguous'
    elif ambiguity in ['unambiguous', 'long unambiguous']:
        if gender == 'male':
            label = 'unambiguous_male'
        elif gender == 'female':
            label = 'unambiguous_female'
        else:
            label = 'ambiguous'

    new_dict_i_need = {'seed': entry['seed'], 'sentence': sent, 'label': label, 'stereotype': entry['stereotype']}
    unique_sents.add(sent)
    dataset_i_really_need.append(new_dict_i_need)

print(dataset_i_really_need[75])

{'seed': 'attendant', 'sentence': 'A flight attendant must handle demanding passengers while remaining professional.', 'label': 'ambiguous', 'stereotype': 'female-stereotyped'}


In [14]:
# creating splits and writing to tsv

random.seed(14)
random.shuffle(dataset_i_really_need)

total_n = len(dataset_i_really_need)
dev_n = int(total_n * 0.1)
test_n = int(total_n * 0.4)

dev_set = dataset_i_really_need[:dev_n]
test_set = dataset_i_really_need[dev_n:test_n]
train_set = dataset_i_really_need[test_n:total_n+1]

with open(fairtranslate_dev, 'w') as f:
    f.write('seed\tsentence\tlabel\tstereotype\n')
    for entry in dev_set:
        sentence = entry['sentence']
        label = entry['label']
        stereotype = entry['stereotype']
        seed = entry['seed']
        f.write(seed + '\t' + sentence + '\t' + label + '\t' + stereotype + '\n')

with open(fairtranslate_test, 'w') as f:
    f.write('seed\tsentence\tlabel\tstereotype\n')
    for entry in test_set:
        sentence = entry['sentence']
        label = entry['label']
        stereotype = entry['stereotype']
        seed = entry['seed']
        f.write(seed + '\t' + sentence + '\t' + label + '\t' + stereotype + '\n')

with open(fairtranslate_train, 'w') as f:
    f.write('seed\tsentence\tlabel\tstereotype\n')
    for entry in train_set:
        sentence = entry['sentence']
        label = entry['label']
        stereotype = entry['stereotype']
        seed = entry['seed']
        f.write(seed + '\t' + sentence + '\t' + label + '\t' + stereotype + '\n')

## GeNTE

Since the seed nouns are not given here either, they need to be extracted again.

In [15]:
def is_person(word):
    synsets = wn.synsets(word, pos=wn.NOUN)
    for syn in synsets:
        for hypernym in syn.closure(lambda s: s.hypernyms()):
            if hypernym == wn.synset('person.n.01'):
                return True
    return False

In [16]:
gente_lines = []
gente_dataset = []


with open(gente_path, 'r') as infile:
    for line in infile:
        gente_dict = dict()
        row = line.strip('\n').split('\t')
        gente_lines.append(row)
        gente_dict['sentence'] = row[3]
        if row[-1] == 'F':
            gente_dict['label'] = 'unambiguous_female'
        elif row[-1] == 'M':
            gente_dict['label'] = 'unambiguous_male'
        else:
            gente_dict['label'] = 'ambiguous'

        gente_dataset.append(gente_dict)
        
print(gente_lines[1])
print()
print(gente_dataset[900])

['1', 'ep-en-it-10000', 'Set-G', 'I think that this is a positive message that the President-in-Office of the Council must include in her speech, because we still have time to include the Charter in Article 6 of the Treaties.', "Credo che si tratti di un messaggio positivo, che la Presidente del Consiglio deve includere nel suo discorso, in quanto siamo ancora in tempo per inserire la Carta nell'articolo 6 dei Trattati.", "Credo che si tratti di un messaggio positivo, che la Presidenza del Consiglio deve includere nel suo discorso, in quanto siamo ancora in tempo per inserire la Carta nell'articolo 6 dei Trattati.", 'no', 'F']

{'sentence': 'We Liberals, along with certain other members of this House, would like to take this opportunity to make it absolutely crystal clear that we are in favour of improving this openness regulation.', 'label': 'ambiguous'}


In [17]:
for dct in gente_dataset:
    dct['seed'] = 'NOTHING'
    for token in nlp(dct['sentence']):
        if is_person(token.lemma_) and (token.pos_ == 'NOUN' or token.pos_ == 'PROPN'):
            dct['seed'] = token.lemma_
            break

#print(gente_dataset)

with open(gente_full, 'w') as f:
    f.write('seed\tsentence\tlabel\n')
    for entry in gente_dataset:
        sentence = entry['sentence']
        label = entry['label']
        seed = entry['seed']
        f.write(seed + '\t' + sentence + '\t' + label + '\n')

/home/marie-necker/anaconda3/envs/coralreef/lib/python3.11/site-packages/nltk/corpus/reader/wordnet.py:604: UserWarning: Discarded redundant search for Synset('abstraction.n.06') at depth 5
  for synset in acyclic_breadth_first(self, rel, depth):
/home/marie-necker/anaconda3/envs/coralreef/lib/python3.11/site-packages/nltk/corpus/reader/wordnet.py:604: UserWarning: Discarded redundant search for Synset('entity.n.01') at depth 7
  for synset in acyclic_breadth_first(self, rel, depth):
/home/marie-necker/anaconda3/envs/coralreef/lib/python3.11/site-packages/nltk/corpus/reader/wordnet.py:604: UserWarning: Discarded redundant search for Synset('entity.n.01') at depth 11
  for synset in acyclic_breadth_first(self, rel, depth):
/home/marie-necker/anaconda3/envs/coralreef/lib/python3.11/site-packages/nltk/corpus/reader/wordnet.py:604: UserWarning: Discarded redundant search for Synset('entity.n.01') at depth 10
  for synset in acyclic_breadth_first(self, rel, depth):
/home/marie-necker/anacon

In [18]:
seeds_gente = [
    'Ambassador', 'Chairman', 'Chief', 'Commissioner', 'Councillor',
    'General', 'Member', 'Minister', 'Ombudsman', 'President', 'Secretary',
    'artist', 'author', 'activist', 'actor', 'advocate', 'ancestor',
    'assistant', 'attorney', 'beneficiary', 'breeder', 'broadcaster',
    'businessman', 'candidate', 'champion', 'chairperson', 'chairman',
    'child', 'citizen', 'colleague', 'compatriot', 'consumer', 'coordinator',
    'criminal', 'culprit', 'daughter', 'dealer', 'defendant', 'delegate',
    'democrat', 'demonstrator', 'deputy', 'dietician', 'director', 'distributor',
    'doctor', 'draftsman', 'draftsperson', 'driver', 'economist', 'entrepreneur',
    'expert', 'executive', 'farmer', 'federalist', 'feminist', 'fisherman',
    'friend', 'functionary', 'governor', 'girl', 'grower', 'guardian',
    'historian', 'immigrant', 'inhabitant', 'inspector', 'investor',
    'journalist', 'judge', 'justice', 'labourer', 'lawyer', 'leader',
    'manager', 'manufacturer', 'member', 'militant', 'minister', 'murderer',
    'negotiator', 'observer', 'officer', 'official', 'ombudsman', 'operator',
    'owner', 'parliamentarian', 'parent', 'patient', 'pensioner', 'player',
    'politician', 'president', 'prisoner', 'producer', 'promoter',
    'prosecutor', 'rapporteur', 'reader', 'refugee', 'representative',
    'researcher', 'resident', 'scientist', 'servant', 'sister', 'socialist',
    'son', 'specialist', 'spokesperson', 'student', 'surgeon', 'teacher',
    'technician', 'terrorist', 'thief', 'traveller', 'veteran', 'victim',
    'visitor', 'winner', 'worker', 'woman', 'boy', 'brother', 'man',
    'seaman', 'rapporteur', 'representative', 'official']

In [19]:
gente_dataset_final = []

for dct in gente_dataset:
    seed = None
    seed_found = False
    for token in nlp.tokenizer(dct['sentence']):
        if token.text in seeds_gente:
            seed = token.text
            seed_found = True
            break
    if seed_found:
        dct['seed'] = seed
        gente_dataset_final.append(dct)


            

In [20]:
for d in gente_dataset_final[:10]:
    print(d)

{'sentence': 'I think that this is a positive message that the President-in-Office of the Council must include in her speech, because we still have time to include the Charter in Article 6 of the Treaties.', 'label': 'unambiguous_female', 'seed': 'President'}
{'sentence': 'I welcome this excellent report from my colleague Mrs Skinner.', 'label': 'unambiguous_female', 'seed': 'colleague'}
{'sentence': 'For all these reasons, we, as our chairman, Mrs Poettering, has already said, are going to vote in favour of this Charter.', 'label': 'unambiguous_female', 'seed': 'chairman'}
{'sentence': 'The chairman of the group to which the majority of governments in the European Union belong said quite clearly that she will not approve the reform of the Treaty of Nice if no reference is made in Article 6.', 'label': 'unambiguous_female', 'seed': 'chairman'}
{'sentence': 'My colleague Mrs Armonie Bordes cuttingly pointed out yesterday the little benefits of deploying community capital and pension pre

In [21]:
random.seed(14)
random.shuffle(gente_dataset_final)

total_n = len(gente_dataset_final)
dev_n = int(total_n * 0.1)
test_n = int(total_n * 0.4)

dev_set = gente_dataset_final[:dev_n]
test_set = gente_dataset_final[dev_n:test_n]
train_set = gente_dataset_final[test_n:total_n+1]

with open(gente_dev, 'w') as f:
    f.write('seed\tsentence\tlabel\n')
    for entry in dev_set:
        sentence = entry['sentence']
        label = entry['label']
        seed = entry['seed']
        f.write(seed + '\t' + sentence + '\t' + label + '\n')

with open(gente_test, 'w') as f:
    f.write('seed\tsentence\tlabel\n')
    for entry in test_set:
        sentence = entry['sentence']
        label = entry['label']
        seed = entry['seed']
        f.write(seed + '\t' + sentence + '\t' + label + '\n')

with open(gente_train, 'w') as f:
    f.write('seed\tsentence\tlabel\n')
    for entry in train_set:
        sentence = entry['sentence']
        label = entry['label']
        seed = entry['seed']
        f.write(seed + '\t' + sentence + '\t' + label + '\n')

In [22]:
print('dev set length:', len(dev_set))
print('test set length:', len(test_set))
print('train set length:', len(train_set))
print()
print('total length GeNTE:',len(gente_dataset_final))

dev set length: 80
test set length: 242
train set length: 483

total length GeNTE: 805


## Lexical Gender Extension

In [23]:
#taken from lanugage as data


# The Gutenberg Corpus contains several literary texts from Project Gutenberg.
nltk.download('gutenberg')

# The Inaugural Address Corpus contains the texts of the inaugural addresses of U.S. presidents.
nltk.download('inaugural')

# The Brown Corpus is a standard corpus of American English texts.
nltk.download('brown')

# The Dependency Treebank Corpus contains sentences annotated with their syntactic structure.
nltk.download('dependency_treebank')

# The Punkt tokenizer is used for sentence splitting.
nltk.download('punkt')  # For tokenization

[nltk_data] Downloading package gutenberg to /home/marie-
[nltk_data]     necker/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!
[nltk_data] Downloading package inaugural to /home/marie-
[nltk_data]     necker/nltk_data...
[nltk_data]   Package inaugural is already up-to-date!
[nltk_data] Downloading package brown to /home/marie-
[nltk_data]     necker/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package dependency_treebank to /home/marie-
[nltk_data]     necker/nltk_data...
[nltk_data]   Package dependency_treebank is already up-to-date!
[nltk_data] Downloading package punkt to /home/marie-
[nltk_data]     necker/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

### Sets with Female and Male Lexical Gender

In [24]:
#lexical gender seed nouns taken from https://github.com/ecmonsen/gendered_words/blob/master/gendered_words.json

lex_gender_path = '../../data/data_that_helps/gendered_words.json'

In [25]:
with open(lex_gender_path, 'r') as infile:
    json_genders = json.load(infile)

In [26]:
female_set = set()

for dct in json_genders:
    if dct['gender'] == 'f' and '_' not in dct['word'] and 'cat' != dct['word'] and 'cow' != dct['word'] and 'miss' not in dct['word'] and dct['word'] not in ['she', 'her', 'herself', 'hers'] and dct['word'] != 'negress':
        female_set.add(dct['word'])

print(female_set)

{'granny', 'co-ed', 'flapper', 'mediatrix', 'postmistress', 'jilt', 'shrew', 'instructress', 'b-girl', 'dame', 'millionairess', 'selectwoman', 'ingenue', 'bacchante', 'patroness', 'hag', 'godmother', 'englishwoman', 'harridan', 'midwife', 'deaconess', 'viscountess', 'maid', 'concubine', 'hostess', 'proprietress', 'circe', 'dairymaid', 'donna', 'lady', 'memsahib', 'coloratura', 'ex-wife', 'she-devil', 'poseuse', 'daughter', 'queen', 'stepmother', 'traitress', 'aunt', 'mrs.', 'charwoman', 'sylph', 'gravida', 'cornishwoman', 'mamma', 'slattern', 'sister', 'committeewoman', 'countess', 'housemother', 'villainess', 'chatelaine', 'forewoman', 'ballerina', 'groupie', 'spokeswoman', 'tomboy', 'jewess', 'undoer', 'coiffeuse', 'maenad', 'sorceress', 'aviatrix', 'rosebud', 'foundress', 'gamine', 'marchioness', 'usherette', 'secundigravida', 'foster-daughter', 'uxor', 'amazon', 'bridesmaid', 'quadripara', "light-o'-love", 'curandera', 'bobbysoxer', 'wanton', 'nanny', 'canary', 'soubrette', 'manage

In [27]:
male_set = set()

for dct in json_genders:
    if dct['gender'] == 'm' and '_' not in dct['word'] and 'cat' != dct['word'] and 'cow' != dct['word'] and 'mister' not in dct['word'] and dct['word'] not in ['he', 'him', 'his', 'himself'] and dct['word'] != 'negr0':
        male_set.add(dct['word'])

print(male_set)

{'hangman', 'bull', 'baritone', 'cleric', 'oarsman', 'codger', 'milkman', 'wencher', 'militiaman', 'tarzan', 'freedman', 'liveryman', 'czar', 'knight-errant', 'lumberman', 'pope', 'cameraman', 'townee', 'buddy', 'father-in-law', 'monsignor', 'subdeacon', 'page', 'wittol', 'mafioso', 'caveman', 'showman', 'stiff', 'godfather', 'elector', 'seedsman', 'bacchant', 'loon', 'groomsman', 'imam', 'potboy', 'dayboy', 'letterman', 'wolf', 'handyman', 'excavator', 'ironside', 'fathead', 'monsieur', 'sod', 'doge', 'busboy', 'mikado', 'jawan', 'clergyman', 'roman', 'tallyman', 'layman', 'benedick', 'squire', 'kaiser', 'godson', 'ordinary', 'praetor', 'burgrave', 'fauntleroy', 'vicar-general', 'groundsman', 'augustinian', 'linkboy', 'pointsman', 'pachuco', 'sadhu', 'mailman', 'coalman', 'eparch', 'gasman', 'timberman', 'businessman', 'grandfather', 'archdeacon', 'ganger', 'frenchman', 'bachelor', 'raftsman', 'plainclothesman', 'clansman', 'brother-in-law', 'pitchman', 'lackey', 'klansman', 'great-un

In [28]:
gendered_words = ["he", "his", "her", "she", "him", "man", "women", "men", "woman", "spokesman", "wife", "himself", "son", "mother", "father", "chairman", "daughter", "husband", "guy", "girls", "girl", "boy", "boys", "brother", "spokeswoman", "female", "sister", "male", "herself", "brothers", "dad", "actress", "mom", "sons", "girlfriend", "daughters", "lady", "boyfriend", "sisters", "mothers", "king", "businessman", "grandmother", "grandfather", "deer", "ladies", "uncle", "males", "congressman", "grandson", "bull", "queen", "businessmen", "wives", "widow", "nephew", "bride", "females", "aunt", "prostate cancer", "lesbian", "chairwoman", "fathers", "moms", "maiden", "granddaughter", "younger brother", "lads", "lion", "gentleman", "fraternity", "bachelor", "niece", "bulls", "husbands", "prince", "colt", "salesman", "hers", "dude", "beard", "filly", "princess", "lesbians", "councilman", "actresses", "gentlemen", "stepfather", "monks", "ex girlfriend", "lad", "sperm", "testosterone", "nephews", "maid", "daddy", "mare", "fiance", "fiancee", "kings", "dads", "waitress", "maternal", "heroine", "nieces", "girlfriends", "sir", "stud", "mistress", "lions", "estranged wife", "womb", "grandma", "maternity", "estrogen", "ex boyfriend", "widows", "gelding", "diva", "teenage girls", "nuns", "czar", "ovarian cancer", "countrymen", "teenage girl", "penis", "bloke", "nun", "brides", "housewife", "spokesmen", "suitors", "menopause", "monastery", "motherhood", "brethren", "stepmother", "prostate", "hostess", "twin brother", "schoolboy", "brotherhood", "fillies", "stepson", "congresswoman", "uncles", "witch", "monk", "viagra", "paternity", "suitor", "sorority", "macho", "businesswoman", "eldest son", "gal", "statesman", "schoolgirl", "fathered", "goddess", "hubby", "stepdaughter", "blokes", "dudes", "strongman", "uterus", "grandsons", "studs", "mama", "godfather", "hens", "hen", "mommy", "estranged husband", "elder brother", "boyhood", "baritone", "grandmothers", "grandpa", "boyfriends", "feminism", "countryman", "stallion", "heiress", "queens", "witches", "aunts", "semen", "fella", "granddaughters", "chap", "widower", "salesmen", "convent", "vagina", "beau", "beards", "handyman", "twin sister", "maids", "gals", "housewives", "horsemen", "obstetrics", "fatherhood", "councilwoman", "princes", "matriarch", "colts", "ma", "fraternities", "pa", "fellas", "councilmen", "dowry", "barbershop", "fraternal", "ballerina"]

obvious_gender = ['female', 'male', 'woman', 'man', 'girl', 'boy', 'sister', 'brother', 'daughter', 'son', 'grandmother', 'grandfather', 'wife', 'husband', 'lady']

In [29]:
#extreme_she_occupations = ["homemaker", "nurse", "receptionist", "librarian", "socialite", "hairdresser", "nanny", "bookkeeper", "stylist", "housekeeper"]

#extreme_he_occupations = ["maestro", "skipper", "protege", "philosopher", "captain", "architect", "financier", "warrior", "broadcaster", "magician"]

#neutral_occupations = []

#print(gendered_words)
#print()

#with open('mturk_stereotypes.csv') as infile:
 #   for line in infile:
  #      row = line.strip().split(',')
   #     #print(row[1])
    #    #print(row[0])
     #   if row[1] != 'stereotype_score' and row[0] not in gendered_words:
      #      #print(float(row[1]))
       #     if float(row[1]) < 1.0:
        #        extreme_he_occupations.append(row[0])
         #   elif float(row[1]) >= 3.0:
          #      extreme_she_occupations.append(row[0])
           # elif float(row[1]) < 2.5 and float(row[1]) > 1.5:
            #    neutral_occupations.append(row[0])


#print(set(extreme_he_occupations))
#print()
#print(set(extreme_she_occupations))
#print()
#print(set(neutral_occupations))

In [30]:
extreme_he_occupations = {
    'taxi_driver', 'footballer', 'soldier', 'financier', 'mobster', 'pastor',
    'actor', 'warrior', 'president', 'baron', 'bishop', 'major_leaguer', 'ballplayer',
    'welder', 'captain', 'priest', 'custodian', 'sportswriter', 'architect', 'dean',
    'philosopher', 'magician', 'gangster', 'bodyguard', 'barber', 'janitor', 'warden',
    'preacher', 'maestro', 'fighter_pilot', 'midfielder', 'butler', 'butcher',
    'mechanic', 'trucker', 'laborer'
}

extreme_she_occupations = {
    'librarian', 'receptionist', 'homemaker', 'nanny', 'secretary', 'housekeeper', 'socialite', 'nurse', 'dancer',
    'stylist', 'hairdresser'
}

neutral_occupations = {
    'envoy', 'strategist', 'tutor', 'poet', 'provost', 'author', 'graphic_designer',
    'gardener', 'planner', 'entertainer', 'councilor', 'civil_servant', 'vocalist',
    'doctoral_student', 'naturalist', 'sociologist', 'guidance_counselor',
    'correspondent', 'analyst', 'singer', 'teenager', 'observer', 'researcher',
    'jurist', 'accountant', 'swimmer', 'anthropologist', 'pathologist', 'pharmacist',
    'musician', 'missionary', 'attorney', 'consultant', 'mediator', 'administrator',
    'promoter', 'commentator', 'psychologist', 'protester', 'pollster', 'undersecretary',
    'lyricist', 'trumpeter', 'valedictorian', 'performer', 'artiste', 'photographer',
    'pundit', 'journalist', 'novelist', 'cellist', 'narrator', 'worker', 'prosecutor',
    'associate_dean', 'comedian', 'campaigner', 'negotiator', 'bartender', 'illustrator',
    'chemist', 'pianist', 'advocate', 'student', 'publicist', 'geologist', 'instructor',
    'psychiatrist', 'citizen', 'dermatologist', 'artist', 'collector', 'pediatrician',
    'photojournalist', 'biologist', 'epidemiologist', 'curator', 'columnist', 'servant',
    'freelance_writer', 'aide', 'associate_professor', 'counselor', 'parishioner',
    'philanthropist', 'baker', 'environmentalist', 'painter', 'treasurer',
    'singer_songwriter', 'lecturer', 'employee', 'writer', 'radiologist', 'restaurateur'
}

In [31]:
def extension_dataset(occupation_set, label):
    entries = []
    sources = load_corpus_sources()
    for src_label, text in sources:
        text_clean = ' '.join(text.split())
        chunks = small_enough_for_spacy(text_clean)
        mentioned = set()
        label_count = 0
        for chunk in chunks:
            spacy_doc = nlp(chunk)
            for sent in spacy_doc.sents:
                has_pronoun = False
                for token in sent:
                    if token.lemma_ in {'he', 'she', 'him', 'her', 'his', 'hers', 'himself', 'herself'}:
                        has_pronoun = True
                        break
                if has_pronoun:
                    continue
                has_person = False
                for ent in sent.ents:
                    if ent.label_ == 'PERSON':
                        has_person = True
                        break
                if has_person:
                    continue
                #has_gendered = False
                #for token in sent:
                #    if token.lemma_ in gendered_words:
                #        has_gendered = True
                #        break
                #if has_gendered:
                #    continue
                for token in sent:
                    if token.pos_ == 'NOUN' and token.text in occupation_set and token.text not in mentioned:
                        seed = token.text.strip()
                        sentence = clean_sentence(sent.text)

                        if seed == '' or sentence == '':
                            continue

                        mentioned.add(token.text)
                        label_count += 1
                        entries.append({'seed': seed, 'sentence': sentence, 'label': label, 'stereotype': '-'})
                        break
        print(f"[{src_label}] {label_count} sentences, nice.")
    print(f"\n{label}: {len(entries)} sentences in total!!")
    return entries




In [32]:
def stereotype_dataset(occupation_set, label, stereotype):
    entries = []
    sources = load_corpus_sources()
    for src_label, text in sources:
        text_clean = ' '.join(text.split())
        chunks = small_enough_for_spacy(text_clean)
        mentioned = set()
        label_count = 0
        for chunk in chunks:
            spacy_doc = nlp(chunk)
            for sent in spacy_doc.sents:
                has_pronoun = False
                for token in sent:
                    if token.lemma_ in {'he', 'she', 'him', 'her', 'his', 'hers', 'himself', 'herself'}:
                        has_pronoun = True
                        break
                if has_pronoun:
                    continue
                has_person = False
                for ent in sent.ents:
                    if ent.label_ == 'PERSON':
                        has_person = True
                        break
                if has_person:
                    continue
                has_gendered = False
                for token in sent:
                    if token.lemma_ in gendered_words:
                        has_gendered = True
                        break
                if has_gendered:
                    continue
                for token in sent:
                    if token.pos_ == 'NOUN' and token.text in occupation_set and token.text not in mentioned:
                        seed = token.text.strip()
                        sentence = clean_sentence(sent.text)

                        if seed == '' or sentence == '':
                            continue

                        mentioned.add(token.text)
                        label_count += 1
                        entries.append({'seed': seed, 'sentence': sentence, 'label': label, 'stereotype': stereotype})
                        break
        print(f"[{src_label}] {label_count} sentences, nice.")
    print(f"\n{label}/{stereotype}: {len(entries)} sentences in total!!")
    return entries

#female_entries = extension_dataset(female_set, "unambiguous_female")
#print()
#male_entries = extension_dataset(male_set, "unambiguous_male")
#print()
#extreme_he_entries = stereotype_dataset(extreme_he_occupations, "ambiguous", "m")
#print()
#extreme_she_entries = stereotype_dataset(extreme_she_occupations, "ambiguous", "f")
#print()
#neutral_entries = stereotype_dataset(neutral_occupations, "ambiguous", "neutral")

#random.shuffle(male_entries)
#male_entries = male_entries[:500]
#random.shuffle(extreme_he_entries)
#extreme_he_entries = extreme_he_entries[:500]
#random.shuffle(extreme_she_entries)
#extreme_she_entries = extreme_she_entries[:500]
#random.shuffle(neutral_entries)
#neutral_entries = neutral_entries[:500]

#extension_entries = female_entries + male_entries + extreme_he_entries + extreme_she_entries + neutral_entries
#random.shuffle(extension_entries)

#total_n = len(extension_entries)
#dev_n = int(total_n * 0.2)
#test_n = int(total_n * 0.5)
#dev_set = extension_entries[:dev_n]
#test_set = extension_entries[dev_n:test_n]
#train_set = extension_entries[test_n:total_n+1]

#with open(lexical_dev, 'w') as f:
 #   f.write('seed\tsentence\tlabel\tstereotype\n')
  #  for entry in dev_set:
   #     sentence = entry['sentence']
    #    label = entry['label']
     #   seed = entry['seed']
      #  stereotype = entry['stereotype']
       # f.write(seed + '\t' + sentence + '\t' + label + '\t' + stereotype + '\n')

#with open(lexical_test, 'w') as f:
 #   f.write('seed\tsentence\tlabel\tstereotype\n')
  #  for entry in test_set:
   #     sentence = entry['sentence']
    #    label = entry['label']
     #   seed = entry['seed']
      #  stereotype = entry['stereotype']
       # f.write(seed + '\t' + sentence + '\t' + label + '\t' + stereotype + '\n')

#with open(lexical_train, 'w') as f:
 #   f.write('seed\tsentence\tlabel\tstereotype\n')
  #  for entry in train_set:
   #     sentence = entry['sentence']
    #    label = entry['label']
     #   seed = entry['seed']
      #  stereotype = entry['stereotype']
       # f.write(seed + '\t' + sentence + '\t' + label + '\t' + stereotype + '\n')

In [33]:
glitter_dev_rows = load_glitter(glitter_dev)
fairtranslate_dev_rows = load_fairtranslate(fairtranslate_dev)
gente_dev_rows = load_gente(gente_dev)
lexical_dev_rows = load_lexical(lexical_dev)

In [34]:
print(glitter_dev_rows[0])

{'seed': 'settlers', 'prev_context': 'However, by RV 10.13.4, Yama is stated to have chosen to leave offspring, but Yamī is not mentioned. Vedic literature states that Yama is the first mortal, and that he chose to die, and then proceeded to create a path to the "other world", where deceased ancestral fathers reside.', 'sentence': 'Due to being the first man to die, he is considered the chief of the dead, lord of settlers, and a father.', 'next_context': 'Throughout the course of Vedic literature, Yama becomes more and more associated with the negative aspects of death and eventually becomes the god of death.', 'label': 'ambiguous'}


In [35]:
male_biased = ["supervisor", "janitor", "cook", "mover", "laborer", "constructor", "chief", "developer", "carpenter", "manager", "lawyer", "farmer", "driver", "salesperson", "physician", "guard", "analyst", "mechanic", "sheriff", "CEO"]

female_biased = ["cashier", "teacher", "nurse", "assistant", "secretary", "auditor", "cleaner", "receptionist", "clerk", "counselor", "designer", "hairdresser", "attendant", "writer", "housekeeper", "baker", "accountant", "editor", "librarian", "tailor"]